# Clase 03 — NumPy y Pandas

**Arrays multidimensionales, vectorización, álgebra lineal y la relación NumPy ↔ Pandas para transformar datos reales.**

Este notebook acompaña la teoría de la Clase 03 con ejemplos ejecutables de cada unidad: repaso de Python, NumPy (ndarray, broadcasting, álgebra lineal), Pandas (Series, DataFrames, preprocesamiento, integración y agregación) y la inspección inicial de un dataset real.

## Bloque 0 — Repaso de la Clase Anterior (Fundamentos de Python)

Antes de arrancar con NumPy, repasamos en un solo bloque los cuatro pilares de Python que vamos a necesitar todo el tiempo: **variables**, **condicionales**, **ciclo for** y **funciones**.

¿Por qué repasarlos justo ahora? Porque en NumPy vamos a comparar constantemente "la forma con bucle" contra "la forma vectorizada", y ese contraste solo se aprovecha si el `for` y la función están frescos en la memoria.

### Variables

Una variable es la etiqueta que le ponemos a un dato en memoria. Python decide el tipo automáticamente al momento de la asignación (**tipado dinámico**): no hace falta declarar `int`, `str`, etc. como en otros lenguajes.

Hasta acá son todos **tipos escalares**: un solo valor por variable. Pero Python también tiene **colecciones**, que agrupan muchos escalares en una sola variable — la más simple es la **lista**. Acá definimos `precios_lista`, que vamos a reutilizar en el ciclo `for` y en la función de más abajo.

In [ ]:
# Variables y tipado dinámico: Python infiere el tipo del dato, no lo declaramos
producto = "Auriculares"
precio = 45.90
stock = 12
en_oferta = True

print(f"{producto} -> tipo: {type(producto)}")
print(f"{precio} -> tipo: {type(precio)}")
print(f"{stock} -> tipo: {type(stock)}")
print(f"{en_oferta} -> tipo: {type(en_oferta)}")

# Una lista es una colección: agrupa muchos escalares en una sola variable
precios_lista = [15, 45, 120, 300, 80, 500, 20]
print(f"{precios_lista} -> tipo: {type(precios_lista)}")
print(f"Primer precio: {precios_lista[0]}")
print(f"Cantidad de precios: {len(precios_lista)}")

### Condicionales

`if` / `elif` / `else` son el bifurcador de caminos: evalúan una condición booleana y deciden qué bloque de código se ejecuta.

Esta vez pedimos el stock por teclado con `input()`, para mostrar en vivo cómo cambia el resultado según el valor que ingreses. Además combinamos condiciones con `and` / `or` para tomar una decisión más parecida a una regla de negocio real: sin stock **siempre** es alerta; con poco stock, solo es alerta si el producto **además** está en oferta (mayor demanda esperada).

In [ ]:
# Pedimos el stock por teclado para ver el condicional "en vivo"
stock = int(input("Ingresá el stock actual: "))

# Combinamos condiciones: sin stock SIEMPRE es alerta;
# con poco stock, solo alerta si el producto está en oferta (or / and)
if stock == 0 or (stock < 5 and en_oferta):
    estado = "Atención: revisar reposición"
else:
    estado = "Stock ok"

print(f"{producto}: {estado}")

### Ciclo for

`for` recorre una colección elemento por elemento. Reutilizamos la lista `precios_lista` que definimos en la sección de Variables — es la herramienta que vamos a comparar todo el tiempo contra la vectorización de NumPy en la próxima unidad.

In [ ]:
for p in precios_lista:
    print(f"Precio: {p}")

### Función que reúne todo

Una función combina variables, condicionales y bucles en un solo bloque reutilizable. Ejemplo: clasificar una lista de precios en categorías de negocio.

In [ ]:
def clasificar_precios(precios):
    """Recorre una lista de precios y cuenta cuántos caen en cada categoría de negocio."""
    resumen = {"Económico": 0, "Medio": 0, "Premium": 0}  # variable: contador por categoría

    for precio in precios:               # ciclo for: recorre cada precio de la lista
        if precio < 50:                   # condicional: define la categoría según el rango
            categoria = "Económico"
        elif precio < 200:
            categoria = "Medio"
        else:
            categoria = "Premium"
        resumen[categoria] += 1           # actualiza el contador de la categoría correspondiente

    return resumen                         # return: entrega el resultado para poder reutilizarlo


reporte = clasificar_precios(precios_lista)
print(reporte)

> Guardate esta función: en la próxima unidad vamos a resolver un problema parecido (transformar una lista completa de precios) con NumPy, sin un solo `for`, y vamos a comparar código y velocidad contra este enfoque tradicional.

## Módulo 1 — Arrays Multidimensionales y Vectorización en NumPy

**NumPy es una librería de Python** (no viene incluida por defecto: se instala con `pip install numpy` y se importa con `import numpy as np`). Por dentro está escrita mayormente en C, así que cuando operamos sobre un array, el trabajo pesado lo hace código compilado — no un bucle interpretado de Python. Eso, sumado a que todos los elementos de un array son del **mismo tipo** (homogeneidad), es lo que la hace tan buena para la parte matemática: memoria contigua + operaciones en bloque = hasta 100x más rápido que un `for`.

### El ndarray y la homogeneidad de tipo

A diferencia de una lista de Python (que puede mezclar tipos sin problema), un `ndarray` de NumPy **fuerza a que todos los elementos sean del mismo tipo**. Si mezclás tipos al crearlo, NumPy los "sube" todos al tipo más general para mantener la homogeneidad.

In [ ]:
import numpy as np

lista_python = [1, "dos", 3.0, True]      # una lista mezcla tipos sin problema
array_numpy = np.array([1, 2, 3.0, 4])    # NumPy homogeneiza: todo termina en float64

print(f"lista:  {lista_python}")
print(f"array:  {array_numpy} -> dtype: {array_numpy.dtype}")

### Dimensiones, forma y tipo: `ndim`, `shape`, `dtype`

- `ndim`: cuántos "ejes" tiene el array (0D escalar, 1D vector, 2D matriz, 3D tensor).
- `shape`: cuántos elementos hay en cada eje, como tupla.
- `dtype`: de qué tipo son los elementos (`int64`, `float64`, `bool`...).

### Creación de arrays

`np.array()` convierte una lista existente; `np.zeros()` / `np.ones()` preparan espacio; `np.arange(inicio, fin, paso)` genera secuencias, como el `range()` de Python pero devolviendo un array.

In [ ]:
vector = np.array([10, 20, 30])
print(f"vector: {vector} -> ndim {vector.ndim}, shape {vector.shape}, dtype {vector.dtype}")

matriz = np.array([[1, 2, 3], [4, 5, 6]])
print(f"matriz shape {matriz.shape}, ndim {matriz.ndim}")

print(np.zeros((2, 3)))    # matriz 2x3 de ceros
print(np.ones(4))          # [1. 1. 1. 1.]
print(np.arange(0, 10, 2)) # [0 2 4 6 8]

### Ejemplo real: cargando `stocks.csv` como matriz de NumPy

`stocks.csv` (en esta carpeta) tiene precios mensuales de 14 acciones entre 2016 y 2021. La primera columna es texto (fechas); las otras 14 son precios. Cargamos **solo las columnas numéricas** con `np.genfromtxt`, excluyendo la fecha con `usecols`.

Si incluyéramos la fecha, NumPy — fiel a la regla de homogeneidad — convertiría **todos** los precios a texto para poder mezclarlos. Esa limitación es justamente el motivo por el que en el Módulo 5 vamos a introducir Pandas.

In [ ]:
# Las columnas del CSV son: formatted_date, MCD, SBUX, GOOG, AMZN, MSFT, JPM, BAC, C, MAR, HLT, RCL, V, MA, PYPL
# skip_header=1 salta la fila de encabezados; usecols=range(1, 15) toma las 14 columnas de precios
precios_matriz = np.genfromtxt("stocks.csv", delimiter=",", skip_header=1, usecols=range(1, 15))

print(f"Shape: {precios_matriz.shape}")   # (71, 14) -> 71 meses, 14 acciones
print(f"ndim:  {precios_matriz.ndim}")
print(f"dtype: {precios_matriz.dtype}")

### Reshape y operaciones elemento a elemento

`reshape` reorganiza un array sin tocar sus datos (el total de elementos debe coincidir). Las operaciones aritméticas ocurren "posición a posición", sin bucles.

In [ ]:
# Columna de MSFT (índice 4) como vector 1D
msft = precios_matriz[:, 4]
print(f"msft shape: {msft.shape}")

# reshape a matriz de 71 filas x 1 columna (mismos datos, otra forma)
msft_columna = msft.reshape(71, 1)
print(f"msft_columna shape: {msft_columna.shape}")

# Operaciones elemento a elemento, sin bucles
precio_en_pesos = msft * 1000                  # simulando un tipo de cambio fijo
diferencia_mensual = msft[1:] - msft[:-1]      # variación mes a mes
print(diferencia_mensual[:5])

### Vectorización frente a bucles: retomando el Bloque 0

En el Bloque 0 resolvimos "transformar una lista de precios" con un `for`. Ahora resolvamos el mismo tipo de problema — aplicar un 21% de aumento a los precios de MSFT — de las dos formas, para comparar código y tiempo.

In [ ]:
import time

msft_lista = list(msft)   # los mismos datos, pero como lista de Python

# --- Enfoque tradicional: bucle for ---
inicio = time.time()
msft_aumentado_lista = []
for precio in msft_lista:
    msft_aumentado_lista.append(precio * 1.21)
tiempo_for = time.time() - inicio

# --- Enfoque NumPy: vectorizado ---
inicio = time.time()
msft_aumentado_array = msft * 1.21
tiempo_numpy = time.time() - inicio

print(f"Bucle for : {tiempo_for:.6f}s  ({len(msft_lista)} elementos, 4 líneas de código)")
print(f"NumPy     : {tiempo_numpy:.6f}s  (1 línea de código)")

> **Errores comunes**: mezclar tipos (`np.array([1, "2"])` convierte todo a texto), confundir `shape` con `len()` (en una matriz `len()` solo cuenta filas), e intentar un `reshape` cuyo producto de dimensiones no coincide con el total de elementos. Ver la tabla completa en el `README.md`.

## Módulo 2 — Broadcasting y Operaciones sobre Matrices

### La Matriz de Datos

En `precios_matriz` cada **fila** es un mes (observación) y cada **columna** es una acción (variable). `precios_matriz[i, j]` accede a la fila `i`, columna `j` (indexación base 0).

In [ ]:
# Tercera fila (mes con índice 2), segunda columna (SBUX, índice 1)
print(precios_matriz[2, 1])

### Broadcasting: operar arrays de distinta forma sin bucles

NumPy compara las formas de derecha a izquierda: son compatibles si son iguales o si una de ellas es 1. Ejemplo real: restar la media de cada acción (shape `(14,)`) a toda la matriz de precios (shape `(71, 14)`), sin crear 71 copias del vector de medias.

In [ ]:
medias_por_accion = precios_matriz.mean(axis=0)     # shape (14,) -> una media por columna
print(f"Shape de las medias: {medias_por_accion.shape}")

precios_centrados = precios_matriz - medias_por_accion   # broadcasting: (71,14) - (14,) -> (71,14)
print(precios_centrados[0])   # desviación de cada acción respecto a su propia media, mes 0

### Multiplicación elemento a elemento (`*`) vs. Transposición (`.T`)

`*` multiplica posición a posición (requiere broadcasting compatible). `.T` gira la matriz — filas pasan a ser columnas — sin copiar datos; es la herramienta para alinear dimensiones antes de una multiplicación matricial.

In [ ]:
volatilidad = precios_matriz.std(axis=0)                  # desvío estándar por acción, shape (14,)
precios_normalizados = precios_centrados / volatilidad     # * y / también son elemento a elemento

print(f"precios_matriz:        {precios_matriz.shape}")
print(f"precios_matriz.T:      {precios_matriz.T.shape}")  # (14, 71) -> cada fila ahora es una acción

### Multiplicación matricial (`@`): valor de un portafolio

`@` sigue la regla clásica del álgebra lineal: columnas de A deben igualar filas de B. Con un vector de **pesos** (una proporción por acción), `precios_matriz @ pesos` da el valor del portafolio en cada uno de los 71 meses, en una sola operación.

In [ ]:
# Un peso igual para las 14 acciones (suman 1 entre todas)
pesos = np.ones(14) / 14
print(f"Shape de pesos: {pesos.shape}")

valor_portafolio = precios_matriz @ pesos      # (71,14) @ (14,) -> (71,)
print(f"Shape del resultado: {valor_portafolio.shape}")
print(valor_portafolio[:5])   # valor del portafolio en los primeros 5 meses

> **Error común de broadcasting**: `precios_matriz - np.array([1, 2, 3])` lanza `ValueError: operands could not be broadcast together with shapes (71,14) (3,)`, porque 3 no es 14 ni es 1. Y confundir `*` con `@`: `*` ajusta cada valor por separado, `@` combina filas y columnas (álgebra lineal).

## Módulo 3 — Álgebra Lineal con NumPy

### Producto punto: retorno esperado de un portafolio

El producto punto toma dos vectores de la misma longitud y devuelve un escalar (multiplica y suma). Con el retorno promedio de cada acción y los pesos del portafolio, obtenemos el retorno esperado del portafolio completo en una sola cuenta.

In [ ]:
retornos_mensuales = (precios_matriz[1:] - precios_matriz[:-1]) / precios_matriz[:-1]  # % de cambio mes a mes
retorno_promedio_por_accion = retornos_mensuales.mean(axis=0)   # shape (14,)

retorno_esperado_portafolio = np.dot(retorno_promedio_por_accion, pesos)   # escalar
print(f"Retorno mensual esperado del portafolio: {retorno_esperado_portafolio:.4%}")

### Sistemas de ecuaciones lineales: `np.linalg.solve`

Para $Ax = b$, calcular la inversa de $A$ es ineficiente y propenso a errores de redondeo. `np.linalg.solve` usa descomposición LU: más rápido y estable (requiere que $A$ sea cuadrada y no singular).

**Ejemplo real**: queremos 100 acciones en total entre MCD y SBUX, con un valor total de 10.000 USD, usando los precios reales del primer mes.

In [ ]:
precio_mcd = precios_matriz[0, 0]    # MCD, mes 0
precio_sbux = precios_matriz[0, 1]   # SBUX, mes 0

# x + y = 100                            (cantidad total de acciones)
# precio_mcd*x + precio_sbux*y = 10000   (valor total en USD)
A = np.array([[1, 1],
              [precio_mcd, precio_sbux]])
b = np.array([100, 10000])

x_mcd, y_sbux = np.linalg.solve(A, b)
print(f"Acciones de MCD:  {x_mcd:.1f}")
print(f"Acciones de SBUX: {y_sbux:.1f}")

### Diagnóstico matricial: `det`, `norm`, `eig`

`np.linalg.det` (determinante, colinealidad), `np.linalg.norm` (magnitud de un vector, distancias) y `np.linalg.eig` (eigenvalues/eigenvectors, base de PCA). Aplicado a la matriz de correlación entre las 14 acciones (cuadrada, 14×14).

In [ ]:
correlaciones = np.corrcoef(precios_matriz.T)   # matriz 14x14: correlación entre cada par de acciones
print(f"Shape: {correlaciones.shape}")

determinante = np.linalg.det(correlaciones)
print(f"Determinante: {determinante:.6f}")   # muy cercano a 0 -> acciones fuertemente correlacionadas

valores_propios, vectores_propios = np.linalg.eig(correlaciones)
print(f"Eigenvalues: {valores_propios.round(2)}")

## Módulo 4 — NumPy y Pandas: la Relación entre el Cálculo y la Estructura de Datos

**Las dos son librerías de Python** (se instalan e importan aparte, no vienen "de fábrica"). NumPy está especializada en la parte **matemática** (números homogéneos, cálculo vectorizado). Pandas está especializada en la parte de **tablas**: columnas con nombre, tipos mixtos (texto + números + fechas conviviendo) y manejo de datos faltantes — justo lo que nos faltó en el Módulo 1, cuando tuvimos que excluir la columna de fechas para poder cargar `stocks.csv` en un `ndarray`.

**La clave que las conecta**: Pandas está construido *sobre* NumPy — cada columna de un DataFrame es, por dentro, un array de NumPy con una etiqueta encima.

In [ ]:
import pandas as pd

# Con Pandas: la tabla completa, fechas incluidas, sin perder nada
df_stocks = pd.read_csv("stocks.csv")
print(df_stocks.dtypes.head())   # formatted_date: object, MCD: float64, SBUX: float64...

# La prueba de que Pandas está construido sobre NumPy:
columna_msft = df_stocks["MSFT"]
print(type(columna_msft))          # pandas.core.series.Series
print(type(columna_msft.values))   # numpy.ndarray -> ¡por dentro, sigue siendo un array!

## Módulo 5 — Introducción a Pandas: Series y DataFrames

### La Serie

Una Serie es una lista "con esteroides": tiene valores **y** un índice (etiqueta por valor). Si no se lo asignamos, Pandas pone `0, 1, 2...` por defecto.

In [ ]:
precios_msft = df_stocks["MSFT"]        # esto ya es una Serie
print(type(precios_msft))
print(precios_msft.index[:5])            # RangeIndex: 0, 1, 2, 3, 4... (por defecto)

# Una Serie creada a mano, con índice con nombre propio (no numérico)
stock_por_sector = pd.Series({"Tecnología": 3, "Consumo": 4, "Finanzas": 3, "Turismo": 3})
print(stock_por_sector["Consumo"])       # acceso directo por etiqueta

### El DataFrame: la tabla bidimensional

Se puede pensar como un diccionario de Series que comparten el mismo índice de filas. Ahora cargamos `stocks.csv` "bien hecho": convertimos la fecha a tipo fecha real (`parse_dates`) y la usamos como índice (`set_index`), en vez de dejarla como columna de texto.

In [ ]:
df_stocks = pd.read_csv("stocks.csv", parse_dates=["formatted_date"])
df_stocks = df_stocks.set_index("formatted_date")

print(df_stocks.index[:3])   # DatetimeIndex: las fechas ahora son el índice, no una columna
print(df_stocks.columns)
print(df_stocks.shape)       # (71, 14) -> mismo tamaño que precios_matriz, pero con etiquetas

### Creación desde listas y diccionarios

No siempre partimos de un archivo. Desde un diccionario (el método más común para un DataFrame), las llaves se convierten en nombres de columna. Vamos a reutilizar `df_sectores` en el Módulo 7 para combinar tablas con `merge`.

In [ ]:
# Desde una lista: la forma más simple, para una Serie
precios_ejemplo = pd.Series([15.0, 45.0, 120.0], name="Precio")

# Desde un diccionario: el método más común para un DataFrame
sectores_dict = {
    "ticker": ["MCD", "SBUX", "GOOG", "AMZN", "MSFT", "JPM", "BAC"],
    "sector": ["Consumo", "Consumo", "Tecnología", "Consumo", "Tecnología", "Finanzas", "Finanzas"],
}
df_sectores = pd.DataFrame(sectores_dict)
print(df_sectores)

> **Errores comunes**: `df["MSFT"]` devuelve una Serie; `df[["MSFT", "AAPL"]]` (doble corchete) devuelve un DataFrame. El índice no siempre es un número (acá son fechas). Y si una columna numérica trae un solo valor de texto, Pandas convierte **toda** la columna a `object` y no se puede calcular hasta limpiarla.

## Módulo 6 — Preprocesamiento de Datos

### Simulando datos ausentes

`stocks.csv` viene limpio, así que simulamos el problema sobre una copia — una práctica habitual para probar un pipeline de limpieza.

In [ ]:
df_sucio = df_stocks.copy()

# Introducimos NaN a propósito para simular datos reales incompletos
df_sucio.iloc[3, 2] = np.nan     # un NaN en la fila 3
df_sucio.iloc[7, 0] = np.nan     # primer NaN en la fila 7
df_sucio.iloc[7, 5] = np.nan     # segundo NaN en la MISMA fila 7
df_sucio.iloc[20, 1] = np.nan    # un NaN en la fila 20

print(df_sucio.isnull().sum())
print(f"Total de NaN: {df_sucio.isnull().sum().sum()}")

### El algoritmo de decisión y la Regla de Oro

Regla por fila: **más de 1** NaN → eliminar el registro; **exactamente 1** → imputar con la media de la columna. **Regla de oro**: las medias se calculan *antes* de eliminar filas (evita Data Leakage).

In [ ]:
# 1) Medias ANTES de eliminar nada (regla de oro)
medias_columnas = df_sucio.mean(numeric_only=True)

# 2) Clasificamos cada fila según su cantidad de NaN
nan_por_fila = df_sucio.isnull().sum(axis=1)
filas_a_eliminar = nan_por_fila[nan_por_fila > 1].index
filas_a_imputar = nan_por_fila[nan_por_fila == 1].index

# 3) Aplicamos la regla
df_limpio = df_sucio.drop(index=filas_a_eliminar)
df_limpio = df_limpio.fillna(medias_columnas)

print(f"Filas eliminadas (>1 NaN): {len(filas_a_eliminar)}")
print(f"Filas imputadas (==1 NaN): {len(filas_a_imputar)}")
print(f"NaN restantes: {df_limpio.isnull().sum().sum()}")

## Módulo 7 — Integración, Agregación y Preprocesamiento Avanzado

### Combinar tablas: `melt` + `merge`

`stocks.csv` no trae sectores — eso vive en `df_sectores` (Módulo 5). Primero pasamos `df_stocks` de formato ancho a largo con `melt`, después unimos con `merge` usando `ticker` como clave.

In [ ]:
df_largo = df_stocks.reset_index().melt(
    id_vars="formatted_date", var_name="ticker", value_name="precio"
)
print(df_largo.head())

df_con_sector = pd.merge(
    left=df_largo,
    right=df_sectores,
    on="ticker",
    how="left",              # conservamos TODOS los precios, tengan sector o no
    validate="many_to_one",   # error si "ticker" estuviera duplicado en df_sectores
    indicator=True,
)
print(df_con_sector["_merge"].value_counts())

### Split-Apply-Combine: `agg()` reduce, `transform()` preserva

`.agg()` colapsa cada grupo a una fila con el estadístico. `.transform()` devuelve un valor por cada fila original, proyectando el resultado del grupo — la base de la normalización intragrupo.

In [ ]:
df_con_sector = df_con_sector.dropna(subset=["sector"])   # nos quedamos con las que sí tienen sector

# agg(): reduce a un resumen por sector
resumen_sector = df_con_sector.groupby("sector")["precio"].agg(
    precio_promedio="mean", precio_max="max", n_registros="count"
)
print(resumen_sector)

# transform(): Z-score, pero DENTRO de cada sector (no global)
df_con_sector["precio_z_sector"] = df_con_sector.groupby("sector")["precio"].transform(
    lambda x: (x - x.mean()) / x.std()
)

### Outliers y escalamiento: Winsorización, Z-Score y Robust Scaling

Winsorizar "achata" los extremos a un percentil, en vez de eliminarlos. Z-Score es sensible a outliers; Robust Scaling usa mediana e IQR y es inmune a ellos.

In [ ]:
p1, p99 = np.percentile(df_con_sector["precio"], [1, 99])
precio_winsorizado = np.clip(df_con_sector["precio"], p1, p99)   # los extremos se "achatan" al tope

# Z-Score: sensible a outliers
z_score = (df_con_sector["precio"] - df_con_sector["precio"].mean()) / df_con_sector["precio"].std()

# Robust Scaling: mediana e IQR, inmune a outliers extremos
mediana = df_con_sector["precio"].median()
iqr = df_con_sector["precio"].quantile(0.75) - df_con_sector["precio"].quantile(0.25)
precio_robusto = (df_con_sector["precio"] - mediana) / iqr

print(f"Winsorizado (P1/P99): min={precio_winsorizado.min():.2f}, max={precio_winsorizado.max():.2f}")
print(f"Z-Score:    media={z_score.mean():.2f}, std={z_score.std():.2f}")
print(f"Robusto:    mediana={precio_robusto.median():.2f}")

## Módulo 8 — La Sinergia de Datos: NumPy y Pandas en Profundidad

### Caso completo: volatilidad de un portafolio ($w^T \Sigma w$)

Pandas prepara los datos (retornos porcentuales), NumPy hace la cuenta pesada (covarianza + álgebra lineal) — el motor y la carrocería trabajando juntos.

In [ ]:
# Pandas: preparar los datos (retornos porcentuales, maneja el primer NaN solo)
retornos = df_stocks.pct_change().dropna()

# NumPy: la cuenta pesada (covarianza + álgebra lineal)
matriz_covarianza = np.cov(retornos.values.T)   # retornos.values -> ya es un ndarray
pesos = np.ones(14) / 14

varianza_portafolio = pesos @ matriz_covarianza @ pesos   # w^T Σ w
volatilidad_portafolio = np.sqrt(varianza_portafolio)

print(f"Volatilidad mensual del portafolio: {volatilidad_portafolio:.4%}")

### El mito del bucle for sobre un DataFrame

Recorrer un DataFrame fila por fila con `for` + `.iloc[i]` es lento y destruye la ventaja de las dos librerías. Siempre que exista una operación vectorizada equivalente, se prefiere esa.

In [ ]:
# Mal: recorrer fila por fila
totales = []
for i in range(len(df_stocks)):
    totales.append(df_stocks.iloc[i].sum())

# Bien: vectorizado, con la misma API de Pandas (que delega en NumPy por debajo)
totales_vectorizado = df_stocks.sum(axis=1)

print(totales[:3])
print(totales_vectorizado.head(3))

> **Aplicaciones reales por industria**: Finanzas (nuestro propio `stocks.csv` — series temporales, volatilidad); Retail (`merge` entre "Ventas" e "Inventario" por ID de producto); Investigación científica (NumPy como estándar para procesar imágenes o señales).

## Módulo 9 — Inspección Inicial de Datos y Pre-Entrega

### `head()`, `info()`, `describe()`: el perfilado inicial

Antes de cualquier cálculo, se le "toma el pulso" al dataset: `head()` para ver que cargó bien, `info()` para tipos y nulos, `describe()` para el resumen estadístico.

In [ ]:
print(df_stocks.head())
print(df_stocks.shape)          # (71, 14)
df_stocks.info()
print(df_stocks.describe())
print(df_stocks.isnull().sum()) # en este dataset, todo en cero: no hay nulos reales

### Pre-Entrega: Checkpoint — Estructura Inicial del Dataset

1. **Carga e Inspección**: `pd.read_csv`, `.head()`, `.shape`, `.info()`.
2. **Perfilado Inicial**: `isnull().sum()` y `.describe()`.
3. **Saneamiento y Selección**: al menos 3 filtros booleanos + eliminar alguna columna innecesaria.
4. **Reflexión**: celda Markdown con los problemas encontrados y las variables clave.

**Entregable**: PDF del notebook exportado (código + resultados + comentarios). Nombre sugerido: `Apellido_Nombre_Checkpoint1.pdf`.